# Part 1: Getting Started with hls4ml

In this notebook you will learn the basic workflow to convert a trained neural network into FPGA firmware using hls4ml.

hls4ml is a tool that translates machine learning models (trained in frameworks like Keras/TensorFlow, PyTorch, or scikit-learn) into HLS (High-Level Synthesis) code that can be compiled into FPGA firmware. The goal is to make deploying neural networks on FPGAs accessible without writing hardware description languages like VHDL or Verilog.

We will:
1. Load a small pre-trained dense neural network
2. Convert it to hls4ml with default settings
3. Verify the fixed-point emulation matches the original model
4. Run HLS synthesis and inspect the resource report
5. Use profiling and trace to diagnose precision issues

## Setup

Standard imports.

In [ ]:
from pathlib import Path

from tensorflow.keras.utils import to_categorical
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
import numpy as np
import matplotlib.pyplot as plt
import os
import subprocess

%matplotlib inline
seed = 0
np.random.seed(seed)
import tensorflow as tf
tf.random.set_seed(seed)

def source_settings(settings_path):
    result = subprocess.run(
        ['bash', '-lc', f'source {settings_path} >/dev/null 2>&1 && env'],
        check=True, capture_output=True, text=True,
    )
    os.environ.update(
        line.split('=', 1) for line in result.stdout.splitlines() if '=' in line
    )

if 'vitis_hls' not in os.environ:
    os.environ['PATH'] = '/opt/Xilinx/Vitis_HLS/2024.1/bin:' + os.environ['PATH']

## Load the dataset

We use the wine recognition dataset included with scikit-learn. The task is to classify wine cultivar from 13 chemical measurements such as alcohol, color intensity, and flavonoid content.

We split the data into train and test samples, scale the input features, and one-hot encode the class labels for the Keras model.

In [ ]:
wine = load_wine()
X = wine.data.astype('float32')
y_labels = wine.target
class_names = wine.target_names

X_train_val, X_test, y_train_val_labels, y_test_labels = train_test_split(
    X, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

scaler = StandardScaler()
X_train_val = scaler.fit_transform(X_train_val).astype('float32')
X_test = scaler.transform(X_test).astype('float32')

n_features = X_train_val.shape[1]
n_classes = len(class_names)
y_train_val = to_categorical(y_train_val_labels, n_classes)
y_test = to_categorical(y_test_labels, n_classes)
X_test = np.ascontiguousarray(X_test)

print('Samples:', X.shape[0])
print('Input features:', n_features)
print('Classes:', class_names)

## Define and load the model

We define a compact 2-hidden-layer dense network:

`Input(13) → Dense(16) → ReLU → Dense(8) → ReLU → Dense(3) → Softmax`

This model is intentionally tiny: 387 trainable parameters and only 360 dense multiplications before the Softmax. That keeps the HLS project fast enough for a live tutorial while still showing the full hls4ml workflow.

In [ ]:
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Activation
from tensorflow.keras.regularizers import l1

MODEL_PATH = Path('models/wine_mlp_16x8.keras')
TRAIN_MODEL = False

def make_wine_mlp():
    model = Sequential(name='wine_mlp_16x8')
    model.add(Dense(16, input_shape=(n_features,), name='fc1', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
    model.add(Activation(activation='relu', name='relu1'))
    model.add(Dense(8, name='fc2', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
    model.add(Activation(activation='relu', name='relu2'))
    model.add(Dense(n_classes, name='output', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
    model.add(Activation(activation='softmax', name='softmax'))
    return model

model = make_wine_mlp()
model.summary()
print('Model compute dtype:',model.compute_dtype)

### Load the provided model

For the tutorial we load a pre-trained model so that time is spent on conversion, emulation, profiling, and synthesis. Set `TRAIN_MODEL = True` only if you need to regenerate the provided model file.

The model file should use the same preprocessing and architecture defined above.

In [ ]:
if TRAIN_MODEL:
    MODEL_PATH.parent.mkdir(exist_ok=True)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(
        X_train_val, y_train_val,
        batch_size=16, epochs=40,
        validation_split=0.25, shuffle=True,
    )
    model.save(MODEL_PATH)
else:
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f'{MODEL_PATH} not found. Provide the trained model or set TRAIN_MODEL = True.')
    model = load_model(MODEL_PATH)

model.summary()

### Check performance

Let's establish the floating-point Keras baseline. This is the performance we need to maintain after converting to fixed-point.

In [ ]:
y_keras = model.predict(X_test)
y_keras_labels = np.argmax(y_keras, axis=1)

print('Keras accuracy: {:.4f}'.format(accuracy_score(y_test_labels, y_keras_labels)))
ConfusionMatrixDisplay.from_predictions(
    y_test_labels, y_keras_labels, display_labels=class_names, cmap='Blues', values_format='d'
)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()

---

## Convert to hls4ml

This is where hls4ml comes in. The conversion process has two steps:

1. **Create a configuration**: `config_from_keras_model` generates a dictionary that controls how the model is translated. With `granularity='model'`, it applies the same settings to every layer.
2. **Convert**: `convert_from_keras_model` takes the Keras model and the config, and produces an HLS model.

The default precision is `ap_fixed<16,6>`, which means 16-bit fixed-point numbers with 6 integer bits (including sign). The remaining 10 bits are fractional. This is a common starting point that works well for many models.

We target the ZCU102 evaluation board, which uses the `xczu9eg-ffvb1156-2-e` FPGA part. Resource and latency reports are device-specific, so use the part number for the board you plan to deploy to.

In [ ]:
import hls4ml
import plotting

config = hls4ml.utils.config_from_keras_model(model, granularity='model', backend='Vitis')
print('--- Configuration ---')
plotting.print_dict(config)

hls_model = hls4ml.converters.convert_from_keras_model(
    model, 
    hls_config=config, 
    backend='Vitis',
    output_dir='wine_mlp_16x8/hls4ml_prj', 
    part='xczu9eg-ffvb1156-2-e'
)
hls_model.compile()

### Visualize the HLS model

`plot_model` shows the converted architecture with the data types and shapes at each layer. You can see how hls4ml has assigned `ap_fixed<16,6>` throughout, and how the shapes propagate through the network.

In [ ]:
hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True, to_file=None)

## Bit-accurate emulation

Before spending time on synthesis, we can verify accuracy with a **bit-accurate emulation**. When we call `hls_model.predict()`, hls4ml runs a software simulation that exactly matches what the hardware will compute, using the same fixed-point arithmetic.

This runs on CPU and tells us if the fixed-point precision is sufficient. If accuracy drops significantly, we know we need to adjust the precision before synthesizing.

In [ ]:
hls_model.compile()
y_hls = hls_model.predict(X_test)

y_hls_labels = np.argmax(y_hls, axis=1)

print('Keras  accuracy: {:.4f}'.format(accuracy_score(y_test_labels, y_keras_labels)))
print('hls4ml accuracy: {:.4f}'.format(accuracy_score(y_test_labels, y_hls_labels)))
try:
    for y_keras_single, y_hls4ml_single in zip(y_keras_labels, y_hls_labels):
        np.testing.assert_array_equal(y_keras_single, y_hls4ml_single)
    
    print('\033[1mKeras and hls4ml outputs coincide bit-a-bit!\033[0m')
except AssertionError:
    print('\033[1mMismatch found in Keras and hls4ml outputs!\033[0m')


fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test_labels, y_keras_labels, display_labels=class_names, cmap='Blues', values_format='d', ax=axes[0], colorbar=False
)
axes[0].set_title('Keras')
ConfusionMatrixDisplay.from_predictions(
    y_test_labels, y_hls_labels, display_labels=class_names, cmap='Blues', values_format='d', ax=axes[1], colorbar=False
)
axes[1].set_title('hls4ml')
for ax in axes:
    ax.tick_params(axis='x', labelrotation=30)
plt.tight_layout()

With `ap_fixed<16,6>` the fixed-point hls4ml accuracy should be virtually identical to the floating-point Keras accuracy. 16 bits is generous for this model.

## Synthesize with Vitis HLS

Now we run **C-Synthesis**. Vitis HLS takes the generated C++ code and compiles it into RTL that can be implemented on an FPGA. This produces reports on:

- **Latency**: number of clock cycles for one inference
- **Initiation Interval (II)**: how many cycles before the next inference can start
- **Resource utilization**: how many DSPs, LUTs, FFs, and BRAMs the design uses

In [ ]:
hls_model.build(csim=False)

### Read the synthesis report

In [ ]:
hls4ml.report.read_vivado_report('wine_mlp_16x8/hls4ml_prj/')

### Understanding the report

Key things to look at:

- **DSP usage**: With `ReuseFactor = 1`, the dense layers are fully parallel. Count the dense multiplications: `13×16 + 16×8 + 8×3 = 360`.
- **Latency**: How many clock cycles for one forward pass. At a typical 200 MHz clock, you can compute the actual latency in microseconds.
- **BRAM**: Used for storing weights and the Softmax lookup tables.

If the model didn't fit (e.g., on a smaller FPGA), we would need to either:
- Reduce the precision (fewer bits → smaller multipliers)
- Increase the `ReuseFactor` (reuse DSPs across multiple multiplications, trading throughput for resources)
- Prune the model (set weights to zero, which eliminates multiplications)

---
## Trace

**Trace** shows the actual numerical output of each layer. It is the first debugging tool to reach for when accuracy drops after conversion, because it tells you exactly which layer introduced the largest error.

We compare the output of every layer between Keras (float32) and hls4ml (fixed-point) for the same input.

In [ ]:
from hls4ml.model.profiling import get_ymodel_keras

config_trace = hls4ml.utils.config_from_keras_model(model, granularity='name', backend='Vitis')
for layer in config_trace['LayerName'].keys():
    config_trace['LayerName'][layer]['Trace'] = True

hls_model_trace = hls4ml.converters.convert_from_keras_model(
    model, hls_config=config_trace, backend='Vitis',
    output_dir='wine_mlp_16x8/hls4ml_prj_trace', part='xczu9eg-ffvb1156-2-e'
)

hls_model_trace.compile()
hls4ml_pred, hls4ml_trace = hls_model_trace.trace(X_test[:1000])
keras_trace = get_ymodel_keras(model, X_test[:1000])

print("Keras output of 'fc1', first sample:")
print(keras_trace['fc1'][0])
print()
print("hls4ml output of 'fc1', first sample:")
print(hls4ml_trace['fc1'][0])

You can see small differences between the float32 and fixed-point values. These quantization errors accumulate through the layers. With 16-bit precision the differences are tiny, but if you reduced to e.g. 4 bits, you'd see much larger discrepancies. That's where trace helps you find the culprit.

You could also plot the difference per layer:
```python
for layer in keras_trace:
    diff = np.abs(keras_trace[layer] - hls4ml_trace[layer])
    print(f'{layer}: max diff = {diff.max():.6f}')
```

## Profiling

So far we used the default `ap_fixed<16,6>` precision. But is 16 bits necessary? Could we use 8 bits and save resources? The profiling tool helps answer this.

The `numerical` profiling function plots the distribution of weights and activations against the range representable by the current fixed-point type:
- **Grey boxes** = the range that the fixed-point type can represent
- **Box-and-whisker plots** = the actual distribution of values

If the whiskers extend beyond the grey box, values will saturate or wrap around, which is bad for accuracy. If the grey box is much wider than the data, you're wasting bits.

The hls4ml model must have tracing enabled before activation profiling can run, so this cell uses the trace-enabled model from the previous section.

In [ ]:
from hls4ml.model.profiling import numerical

hls_model_prof = hls_model_trace

# To showcase 8-bit precision, uncomment this block and rerun this cell.
# config_prof = hls4ml.utils.config_from_keras_model(model, granularity='name', backend='Vitis')
# for layer in config_prof['LayerName'].keys():
#     config_prof['LayerName'][layer]['Trace'] = True
#     config_prof['LayerName'][layer]['Precision']['result'] = 'fixed<8,3>'
#
# hls_model_prof = hls4ml.converters.convert_from_keras_model(
#     model, hls_config=config_prof, backend='Vitis',
#     output_dir='wine_mlp_16x8/hls4ml_prj_prof_8bit', part='xczu9eg-ffvb1156-2-e'
# )

profile_figures = numerical(model=model, hls_model=hls_model_prof, X=X_test[:1000])

def share_profile_xaxis(figures):
    axes = [fig.axes[0] for fig in figures if fig is not None and fig.axes]
    if not axes:
        return
    xmin = min(ax.get_xlim()[0] for ax in axes)
    xmax = max(ax.get_xlim()[1] for ax in axes)
    for ax in axes:
        ax.set_xlim(xmin, xmax)

share_profile_xaxis(profile_figures[:2])  # weights before/after optimization
share_profile_xaxis(profile_figures[2:])  # activations before/after optimization

The top plot shows distributions **before** hls4ml optimizations, the bottom after (e.g., after fusing Dense+BatchNorm layers). Look at how tightly the weight distributions fit within the representable range.

With post-training quantization (just reducing bit widths without retraining), around 8 bits is generally the limit before accuracy degrades noticeably. For lower precisions, you'd want quantization-aware training (covered in Part 2).

---

## Summary

The basic hls4ml workflow is:

1. **Load** your trained model in Keras/TensorFlow
2. **Convert** with `hls4ml.converters.convert_from_keras_model`
3. **Verify** accuracy with bit-accurate emulation (`hls_model.predict`)
4. **Synthesize** with `hls_model.build` to get FPGA resource estimates

In the next notebook, we'll apply this workflow to a CNN and explore pruning and quantization to reduce resource usage.

---

## Exercise

From the profiling results, this model looks like a good candidate for 8-bit fixed-point precision. Create a new hls4ml project for the `wine_mlp_16x8` model that uses 8-bit precision.

Start by setting `default_precision='fixed<8,3>'` when generating the hls4ml configuration:

```python
config_8bit = hls4ml.utils.config_from_keras_model(
    model,
    granularity='model',
    backend='Vitis',
    default_precision='fixed<8,3>',
)
```

Then convert the model into a new output directory, run bit-accurate emulation, and compare the 8-bit hls4ml accuracy with the original Keras accuracy.

Finally, synthesize the 8-bit project and compare its latency and resource usage with the original 16-bit project. Which resources changed the most?